# GASP pilot in Google Colab

Runs the LLM pilot (8 scenarios, 5 modes, 2 models, 1 repeat) with no local installation. You need an OpenAI API key; it is typed into a password prompt and kept only in this session's memory. Cost: a few USD. Time: under an hour.

Run the cells top to bottom.

In [ ]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
print('key set (%d characters)' % len(os.environ['OPENAI_API_KEY']))

## Get the code

Either the `seams2027` branch of the repository (once it is pushed), or upload `GASP-seams2027.zip`.

In [ ]:
import subprocess, sys, os
if not os.path.exists('GASP'):
    r = subprocess.run(['git','clone','-q','-b','seams2027','https://github.com/unibuc-cs/GASP.git'], capture_output=True, text=True)
    if r.returncode != 0:
        print('branch not on GitHub yet; upload GASP-seams2027.zip')
        from google.colab import files  # type: ignore
        up = files.upload()
        name = next(iter(up))
        subprocess.run(['unzip','-q','-o',name], check=True)
%cd GASP
!python -m pip install -q -r requirements.txt
!python -m pytest tests -q

## Estimate without API calls

In [ ]:
!python -m gasp.experiments.reproduce --out outputs/paper --no-traces --skip-ablation > /dev/null
!python -m gasp.experiments.run_grid --config configs/llm_grid.yaml --out outputs/llm_estimate --scenarios outputs/paper/scenarios.json --estimate --repeats 1 | tail -2

## Pilot (this is the step that costs money)

In [ ]:
!python -m gasp.experiments.run_grid --config configs/llm_grid.yaml --out outputs/llm_pilot --scenarios outputs/paper/scenarios.json --limit 2 --repeats 1
!python -m gasp.experiments.analyze --episodes outputs/llm_pilot/episodes.csv --out outputs/llm_pilot/stats --pairs M2:M3,M0:M3,M3:M4 --group model > /dev/null
print(open('outputs/llm_pilot/table_llm.md').read())

What to look at: `formatting_failures` well below 1 per episode; `success` in M2 above 0.3; `executed_violations` exactly 0.00 in M3 and M4.

## Download the results and send them back

In [ ]:
!python -m gasp.experiments.pilot_report --pilot outputs/llm_pilot --machine Colab
print(open('outputs/llm_pilot/PILOT_REPORT.md').read())
!zip -q -r llm_pilot.zip outputs/llm_pilot
from google.colab import files  # type: ignore
files.download('llm_pilot.zip')

## Main grid (optional here; better on a machine that can run for hours)

`bash scripts/run_main_grid.sh` does the whole grid with the five modes in parallel and is resumable. In Colab the session may time out; the runner skips finished episodes, so rerunning continues where it stopped.